# YOLO11s-Style Vehicle Detection with ResNet34 Backbone

**EE3703 Group Project Group 6** \
This notebook implements a manual YOLO11s-style vehicle detector in PyTorch using a ResNet34 backbone and Albumentations-based data augmentation.

Key constraints satisfied:
- No black-box Ultralytics training/inference API
- Manual architecture, loss, assignment, metrics, and inference
- Input images resized to **640 × 640**
- Evaluation includes **mAP, Precision, Recall, F1, mAP@0.5:0.95, and Inference Speed**
- Training uses **Albumentations** with bbox-aware augmentation

Dataset structure:
---
```
/kaggle/input/
└── yolo-final-merged/
    └── YOLO Merged Dataset Filtered/
        ├── images/
        │   ├── train/
        │   └── test/
        └── labels/
            ├── train/
            └── test/
```


**Part 0: Load pretrained model (optional)**

In [ ]:
# ── Load a previously saved model version ───────────────────────────────────
# Set this to the path of your saved .pth file from a previous run
# NOTE: this cell is defined early but should only be *executed* after Cell 21
# (imports). If you want to load a checkpoint, run all cells top-to-bottom.
LOAD_PRETRAINED_RUN = None  # <-- set to path string to load, or leave None to train fresh

# Actual loading is deferred to after imports — see the block at the end of Cell 21.

# This notebook now uses a ResNet34 backbone; only load checkpoints trained with the same architecture.


  **Part 1: Setup**

In [ ]:
import os
import time
import math
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter

import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.ops import nms

SEED = 42

def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)

seed_everything(SEED)
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

# GPU Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, flush=True)
print("Seed:", SEED, flush=True)

# Hard stop — never waste a run on CPU
if device.type != "cuda":
    raise RuntimeError(
        "No GPU detected. Go to Settings → Accelerator → GPU T4 x2 and re-run."
    )


In [ ]:
# Ensure Albumentations is available before importing project modules.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("albumentations") is None:
    print("Installing albumentations...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "albumentations"])
else:
    print("albumentations is already installed.")


**Part 2: Paths and folders**

In [ ]:
# ── Explore dataset structure (run this first to confirm paths) ────────────
import os
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 4:
        for f in files[:3]:
            print(f"{indent}  {f}")

print("\n" + "="*50)

# ── Paths ── fixed for Kaggle dataset structure ─────────────────────────────
# Dataset slug: yolo-final-merged
# Subfolder:    YOLO Merged Dataset Filtered
DATA_ROOT = "/kaggle/input/datasets/rishabhshenoy03/yolo-merged-dataset-filtered-final/YOLO Merged Dataset Filtered Final"

# Verify DATA_ROOT exists; if not, auto-detect from /kaggle/input
if not os.path.isdir(DATA_ROOT):
    import glob
    candidates = (
        glob.glob("/kaggle/input/datasets/*/*/*/images") +
        glob.glob("/kaggle/input/datasets/*/*/images") +
        glob.glob("/kaggle/input/*/*/images") +
        glob.glob("/kaggle/input/*/images")
    )
    if candidates:
        DATA_ROOT = os.path.dirname(candidates[0])
        print("Auto-detected DATA_ROOT:", DATA_ROOT)
    else:
        raise RuntimeError("Could not find dataset under /kaggle/input. Check your dataset is attached.")
else:
    print("DATA_ROOT:", DATA_ROOT)

TRAIN_IMG_DIR = os.path.join(DATA_ROOT, "images", "train")
TRAIN_LBL_DIR = os.path.join(DATA_ROOT, "labels", "train")
TEST_IMG_DIR  = os.path.join(DATA_ROOT, "images", "test")
TEST_LBL_DIR  = os.path.join(DATA_ROOT, "labels", "test")

PROJECT_ROOT = "/kaggle/working/YOLO_Project"
os.makedirs(PROJECT_ROOT, exist_ok=True)

for sub in [
    "models", "data", "training", "inference", "gui",
    "outputs", "outputs/checkpoints", "outputs/figures",
    "outputs/predictions", "outputs/logs", "outputs/blind_test_analysis"
]:
    os.makedirs(os.path.join(PROJECT_ROOT, sub), exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Train images:", TRAIN_IMG_DIR)
print("Test images :", TEST_IMG_DIR)

# Verify folders exist
for path in [TRAIN_IMG_DIR, TRAIN_LBL_DIR, TEST_IMG_DIR, TEST_LBL_DIR]:
    exists = os.path.isdir(path)
    count  = len(os.listdir(path)) if exists else 0
    print(f"  {'OK' if exists else 'MISSING'}: {path} ({count} files)")


**Part 3: Class names and sanity check dataset files**

In [ ]:
# Change the order to exactly match YOLO label export IDs.

CLASS_NAMES = {
    0: "Bus",
    1: "Car",
    2: "Motorcycle",
    3: "Truck"
}
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 640

# Centralized evaluation settings
EVAL_CONF_THRESH = 0.45
EVAL_NMS_IOU = 0.40

def list_images(folder):
    return sorted([f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))])

train_images = list_images(TRAIN_IMG_DIR)
test_images = list_images(TEST_IMG_DIR)

print("Train images:", len(train_images))
print("Test images :", len(test_images))
print("Eval conf thresh:", EVAL_CONF_THRESH)
print("Eval NMS IoU   :", EVAL_NMS_IOU)

assert len(train_images) > 0, "No train images found."
assert len(test_images) > 0, "No test images found."


**Part 4: Write necessary .py files if those files not in folder**

In [ ]:
%%writefile /kaggle/working/YOLO_Project/models/modules.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBNAct(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, p=None, act=True):
        super().__init__()
        if p is None:
            p = k // 2
        self.conv = nn.Conv2d(c1, c2, k, s, p, bias=False)
        self.bn   = nn.BatchNorm2d(c2)
        self.act  = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Bottleneck(nn.Module):
    def __init__(self, c, shortcut=True, e=0.5):
        super().__init__()
        hidden = int(c * e)
        self.cv1 = ConvBNAct(c, hidden, k=1, s=1)
        self.cv2 = ConvBNAct(hidden, c, k=3, s=1)
        self.use_shortcut = shortcut

    def forward(self, x):
        y = self.cv2(self.cv1(x))
        return x + y if self.use_shortcut else y


class C3k2(nn.Module):
    """
    Cross-Stage Partial bottleneck with 2 convolutions.
    Faithful to YOLO11 C3k2 (inherits C2f pattern):
      cv1 projects to 2*c_, output split into bypass + processing halves.
      n Bottleneck blocks run sequentially, each on the previous output.
      All tensors (bypass + n block outputs) are concatenated then projected.
    Channel check (e=0.5 default):
      c_ = c2//2
      cv1: c1 -> 2*c_
      concat: (2+n)*c_ channels
      cv2: (2+n)*c_ -> c2
    """
    def __init__(self, c1, c2, n=2, shortcut=True, e=0.5):
        super().__init__()
        self.c_ = int(c2 * e)
        self.cv1 = ConvBNAct(c1, 2 * self.c_, k=1, s=1)
        self.cv2 = ConvBNAct((2 + n) * self.c_, c2, k=1, s=1)
        self.m   = nn.ModuleList(
            Bottleneck(self.c_, shortcut=shortcut, e=1.0) for _ in range(n)
        )

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, dim=1))   # [bypass_half, proc_half]
        y.extend(m(y[-1]) for m in self.m)       # append each block output
        return self.cv2(torch.cat(y, dim=1))


class SPPF(nn.Module):
    """Spatial Pyramid Pooling - Fast. Correct: 3 pooling passes, concat 4 tensors."""
    def __init__(self, c1, c2, k=5):
        super().__init__()
        self.cv1  = ConvBNAct(c1, c2, k=1, s=1)
        self.pool = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)
        self.cv2  = ConvBNAct(c2 * 4, c2, k=1, s=1)

    def forward(self, x):
        x  = self.cv1(x)
        y1 = self.pool(x)
        y2 = self.pool(y1)
        y3 = self.pool(y2)
        return self.cv2(torch.cat([x, y1, y2, y3], dim=1))


class PSAAttention(nn.Module):
    """
    Position-Sensitive multi-head self-attention used inside C2PSA.
    Operates on c channels with num_heads heads (head_dim = c // num_heads).
    Uses fused QKV projection then splits, matching YOLO11 PSA implementation.
    """
    def __init__(self, c, num_heads):
        super().__init__()
        assert c % num_heads == 0, f"c ({c}) must be divisible by num_heads ({num_heads})"
        self.num_heads = num_heads
        self.head_dim  = c // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv  = nn.Conv2d(c, 3 * c, 1, bias=False)
        self.proj = nn.Conv2d(c, c, 1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        # Fused QKV: [B, 3*C, H, W] -> [B, 3, heads, head_dim, N]
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, N)
        q, k, v = qkv.unbind(1)              # each [B, heads, head_dim, N]
        q = q.transpose(-2, -1)              # [B, heads, N, head_dim]
        k = k.transpose(-2, -1)
        v = v.transpose(-2, -1)
        attn = torch.softmax(q @ k.transpose(-2, -1) * self.scale, dim=-1)
        out  = (attn @ v).transpose(-2, -1)  # [B, heads, head_dim, N]
        out  = out.reshape(B, C, H, W)
        return self.proj(out)


class C2PSA(nn.Module):
    """
    Cross-Stage Partial with Position-Sensitive Attention.
    Faithful to YOLO11 C2PSA (ultralytics/nn/modules/block.py):
      - cv1 projects c -> 2*(c//2), output split into attention branch (a) and bypass (b)
      - a goes through PSAAttention with residual connection
      - cat([a, b]) projected back to c by cv2
    num_heads = max(1, (c//2) // 64) gives head_dim=64 for c=512 (4 heads).
    """
    def __init__(self, c, e=0.5):
        super().__init__()
        self.c_   = int(c * e)                        # half channels = c//2
        num_heads = max(1, self.c_ // 64)             # head_dim=64
        self.cv1  = ConvBNAct(c, 2 * self.c_, k=1, s=1)
        self.attn = PSAAttention(self.c_, num_heads)
        self.cv2  = ConvBNAct(2 * self.c_, c, k=1, s=1)

    def forward(self, x):
        a, b = self.cv1(x).chunk(2, dim=1)   # a=attention branch, b=bypass
        a = a + self.attn(a)                  # residual attention on half channels
        return self.cv2(torch.cat([a, b], dim=1))


In [ ]:
%%writefile /kaggle/working/YOLO_Project/models/detection_head.py
import torch
import torch.nn as nn
from models.modules import ConvBNAct

class DetectHead(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.out_dim = 4 + 1 + num_classes  # xywh + obj + cls

        self.stems = nn.ModuleList([
            nn.Sequential(
                ConvBNAct(c, c, 3, 1),
                ConvBNAct(c, c, 3, 1),
                nn.Conv2d(c, self.out_dim, 1)
            )
            for c in in_channels
        ])

    def forward(self, feats):
        return [head(f) for head, f in zip(self.stems, feats)]

In [ ]:
%%writefile /kaggle/working/YOLO_Project/models/backbone_resnet.py
import torch
import torch.nn as nn
from torchvision.models import resnet34, ResNet34_Weights

class ResNet34Backbone(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        if pretrained:
            weights = ResNet34_Weights.DEFAULT
            backbone = resnet34(weights=weights)
        else:
            backbone = resnet34(weights=None)

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1   # /4   -> 64 ch
        self.layer2 = backbone.layer2   # /8   -> 128 ch
        self.layer3 = backbone.layer3   # /16  -> 256 ch
        self.layer4 = backbone.layer4   # /32  -> 512 ch

    def forward(self, x):
        x = self.conv1(x)      # /2
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)    # /4

        p2 = self.layer1(x)    # /4
        p3 = self.layer2(p2)   # /8
        p4 = self.layer3(p3)   # /16
        p5 = self.layer4(p4)   # /32

        return p2, p3, p4, p5

# Backward-compatible alias for any older imports inside the notebook.
ResNet18Backbone = ResNet34Backbone


In [ ]:
%%writefile /kaggle/working/YOLO_Project/models/yolo11s.py
import torch
import torch.nn as nn

from models.modules import ConvBNAct, C3k2, SPPF, C2PSA
from models.detection_head import DetectHead
from models.backbone_resnet import ResNet34Backbone

class YOLO11sResNet34(nn.Module):
    def __init__(self, num_classes=4, pretrained_backbone=True):
        super().__init__()

        self.backbone = ResNet34Backbone(pretrained=pretrained_backbone)

        # Highest-level refinement
        self.sppf = SPPF(512, 512)
        self.attn = C2PSA(512)

        # Top-down path
        self.up1 = nn.Upsample(scale_factor=2, mode="nearest")
        self.neck_p4 = C3k2(512 + 256, 256, n=2, shortcut=False)

        self.up2 = nn.Upsample(scale_factor=2, mode="nearest")
        self.neck_p3 = C3k2(256 + 128, 128, n=2, shortcut=False)

        self.up3 = nn.Upsample(scale_factor=2, mode="nearest")
        self.neck_p2 = C3k2(128 + 64, 64, n=2, shortcut=False)

        # Bottom-up path
        self.down_p3 = ConvBNAct(64, 64, 3, 2)
        self.neck_p3_out = C3k2(64 + 128, 128, n=2, shortcut=False)

        self.down_p4 = ConvBNAct(128, 128, 3, 2)
        self.neck_p4_out = C3k2(128 + 256, 256, n=2, shortcut=False)

        self.down_p5 = ConvBNAct(256, 256, 3, 2)
        self.neck_p5_out = C3k2(256 + 512, 512, n=2, shortcut=True)

        # Detect on P2, P3, P4, P5
        self.detect = DetectHead([64, 128, 256, 512], num_classes)

    def forward(self, x):
        p2, p3, p4, p5 = self.backbone(x)

        p5 = self.attn(self.sppf(p5))

        # Top-down
        p4n = self.neck_p4(torch.cat([self.up1(p5), p4], dim=1))
        p3n = self.neck_p3(torch.cat([self.up2(p4n), p3], dim=1))
        p2n = self.neck_p2(torch.cat([self.up3(p3n), p2], dim=1))

        # Bottom-up
        p3o = self.neck_p3_out(torch.cat([self.down_p3(p2n), p3n], dim=1))
        p4o = self.neck_p4_out(torch.cat([self.down_p4(p3o), p4n], dim=1))
        p5o = self.neck_p5_out(torch.cat([self.down_p5(p4o), p5], dim=1))

        return self.detect([p2n, p3o, p4o, p5o])

# Backward-compatible alias for any older notebook references.
YOLO11sResNet18 = YOLO11sResNet34


In [ ]:
%%writefile /kaggle/working/YOLO_Project/data/transforms.py
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _bbox_params(min_visibility=0.1):
    return A.BboxParams(
        format='pascal_voc',
        label_fields=['category_ids'],
        min_visibility=min_visibility
    )

def get_train_transforms():
    return A.Compose([
        A.RandomScale(scale_limit=0.3, p=0.4),
        A.RandomSizedBBoxSafeCrop(height=640, width=640, erosion_rate=0.1, p=0.3),
        A.Resize(640, 640),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.1,
            rotate_limit=5,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.3
        ),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.4),
        A.HueSaturationValue(
            hue_shift_limit=10,
            sat_shift_limit=20,
            val_shift_limit=20,
            p=0.3
        ),
        A.OneOf([
            A.MotionBlur(blur_limit=5, p=1.0),
            A.GaussianBlur(blur_limit=3, p=1.0),
        ], p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ], bbox_params=_bbox_params(min_visibility=0.1))

def get_test_transforms():
    return A.Compose([
        A.Resize(640, 640),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ], bbox_params=_bbox_params(min_visibility=0.0))


In [ ]:
%%writefile /kaggle/working/YOLO_Project/data/dataset.py
import os
import re
import random
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset

def extract_image_number(filename):
    stem = os.path.splitext(os.path.basename(filename))[0]
    match = re.search(r"(\d+)$", stem)
    if match:
        return int(match.group(1))
    digits = re.findall(r"\d+", stem)
    if digits:
        return int(digits[-1])
    return None

def letterbox_with_boxes(img, boxes, new_size=640, color=(114, 114, 114)):
    h0, w0 = img.shape[:2]
    scale = min(new_size / w0, new_size / h0)

    nw = int(round(w0 * scale))
    nh = int(round(h0 * scale))

    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)

    pad_w = new_size - nw
    pad_h = new_size - nh
    left = pad_w // 2
    right = pad_w - left
    top = pad_h // 2
    bottom = pad_h - top

    out = cv2.copyMakeBorder(
        resized, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=color
    )

    new_boxes = []
    for cls, cx, cy, w, h in boxes:
        x1 = (cx - w / 2.0) * w0
        y1 = (cy - h / 2.0) * h0
        x2 = (cx + w / 2.0) * w0
        y2 = (cy + h / 2.0) * h0

        x1 = x1 * scale + left
        x2 = x2 * scale + left
        y1 = y1 * scale + top
        y2 = y2 * scale + top

        x1 = max(0.0, min(float(new_size), x1))
        x2 = max(0.0, min(float(new_size), x2))
        y1 = max(0.0, min(float(new_size), y1))
        y2 = max(0.0, min(float(new_size), y2))

        bw = max(0.0, x2 - x1)
        bh = max(0.0, y2 - y1)
        if bw < 1.0 or bh < 1.0:
            continue

        ncx = ((x1 + x2) / 2.0) / new_size
        ncy = ((y1 + y2) / 2.0) / new_size
        nw_box = bw / new_size
        nh_box = bh / new_size

        new_boxes.append([cls, ncx, ncy, nw_box, nh_box])

    if len(new_boxes) == 0:
        return out, np.zeros((0, 5), dtype=np.float32)
    return out, np.array(new_boxes, dtype=np.float32)

def yolo_to_pascal_voc(boxes, img_w, img_h):
    bboxes = []
    labels = []
    for cls, cx, cy, bw, bh in boxes:
        x1 = (cx - bw / 2.0) * img_w
        y1 = (cy - bh / 2.0) * img_h
        x2 = (cx + bw / 2.0) * img_w
        y2 = (cy + bh / 2.0) * img_h

        x1 = float(np.clip(x1, 0, img_w - 1))
        y1 = float(np.clip(y1, 0, img_h - 1))
        x2 = float(np.clip(x2, 0, img_w - 1))
        y2 = float(np.clip(y2, 0, img_h - 1))

        if x2 <= x1 or y2 <= y1:
            continue

        bboxes.append([x1, y1, x2, y2])
        labels.append(int(cls))
    return bboxes, labels

def pascal_voc_to_yolo(bboxes, labels, img_w, img_h):
    rows = []
    for box, cls in zip(bboxes, labels):
        x1, y1, x2, y2 = box
        x1 = float(np.clip(x1, 0, img_w - 1))
        y1 = float(np.clip(y1, 0, img_h - 1))
        x2 = float(np.clip(x2, 0, img_w - 1))
        y2 = float(np.clip(y2, 0, img_h - 1))

        bw = max(0.0, x2 - x1)
        bh = max(0.0, y2 - y1)
        if bw < 1.0 or bh < 1.0:
            continue

        cx = ((x1 + x2) / 2.0) / img_w
        cy = ((y1 + y2) / 2.0) / img_h
        nw = bw / img_w
        nh = bh / img_h

        rows.append([int(cls), cx, cy, nw, nh])

    if len(rows) == 0:
        return np.zeros((0, 5), dtype=np.float32)
    return np.array(rows, dtype=np.float32)

class YOLODetectionDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, img_size=640, transforms=None, image_names=None, use_letterbox=False, augment=False):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.img_size = img_size
        self.transforms = transforms
        self.use_letterbox = use_letterbox
        self.augment = augment

        all_images = sorted([
            f for f in os.listdir(img_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])
        self.images = sorted(image_names) if image_names is not None else all_images

    def __len__(self):
        return len(self.images)

    def load_labels(self, label_path):
        boxes = []
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    vals = line.strip().split()
                    if len(vals) != 5:
                        continue
                    cls_id, cx, cy, w, h = map(float, vals)
                    boxes.append([int(cls_id), cx, cy, w, h])
        if len(boxes) == 0:
            return np.zeros((0, 5), dtype=np.float32)
        return np.array(boxes, dtype=np.float32)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        lbl_path = os.path.join(self.lbl_dir, os.path.splitext(img_name)[0] + ".txt")

        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Failed to read image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        boxes = self.load_labels(lbl_path)

        if self.use_letterbox:
            img, boxes = letterbox_with_boxes(img, boxes, self.img_size)
        elif self.transforms is None:
            img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)

        if self.transforms is not None:
            img_h, img_w = img.shape[:2]
            pascal_boxes, category_ids = yolo_to_pascal_voc(boxes, img_w, img_h)

            transformed = self.transforms(
                image=img,
                bboxes=pascal_boxes,
                category_ids=category_ids
            )
            img = transformed["image"]

            if hasattr(img, "shape") and len(img.shape) == 3:
                out_h, out_w = int(img.shape[1]), int(img.shape[2])
            else:
                out_h = self.img_size
                out_w = self.img_size

            boxes = pascal_voc_to_yolo(
                transformed["bboxes"],
                transformed["category_ids"],
                out_w,
                out_h
            )
        else:
            img = torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1).float() / 255.0

        target = {
            "boxes": torch.tensor(boxes[:, 1:5], dtype=torch.float32),
            "labels": torch.tensor(boxes[:, 0], dtype=torch.long),
            "image_id": img_name
        }
        return img, target, img_name

def collate_fn(batch):
    imgs, targets, names = zip(*batch)
    return torch.stack(imgs, 0), list(targets), list(names)


In [ ]:
%%writefile /kaggle/working/YOLO_Project/training/assigner.py
import torch


class MultiScaleAssigner:
    """
    4-scale assigner for P2/P3/P4/P5:
      P2 -> stride 4  (tiny)
      P3 -> stride 8  (small)
      P4 -> stride 16 (medium)
      P5 -> stride 32 (large)

    Still one object per cell per scale, but:
    - favors center-near assignments
    - slightly favors smaller objects in collisions
    - allows adjacent-scale assignment near boundaries
    """

    def __init__(
        self,
        num_classes,
        img_size=640,
        strides=(4, 8, 16, 32),
        xsmall_thresh=0.015,
        small_thresh=0.07,     # was 0.05 — gives trucks more P3/P4 signal
        medium_thresh=0.25,    # was 0.18 — routes buses to P4 not just P5
        scale_border_ratio=0.20,
        small_object_bonus=0.20
    ):
        self.num_classes = num_classes
        self.img_size = img_size
        self.strides = strides

        self.xsmall_thresh = xsmall_thresh
        self.small_thresh = small_thresh
        self.medium_thresh = medium_thresh

        self.scale_border_ratio = scale_border_ratio
        self.small_object_bonus = small_object_bonus

    def area_of(self, w, h):
        return float(w * h)

    def choose_scale(self, w, h):
        area = self.area_of(w, h)
        if area < self.xsmall_thresh:
            return 0   # P2
        elif area < self.small_thresh:
            return 1   # P3
        elif area < self.medium_thresh:
            return 2   # P4
        else:
            return 3   # P5

    def candidate_scales(self, w, h):
        area = self.area_of(w, h)
        primary = self.choose_scale(w, h)
        candidates = [primary]

        xs_low = self.xsmall_thresh * (1.0 - self.scale_border_ratio)
        xs_high = self.xsmall_thresh * (1.0 + self.scale_border_ratio)

        s_low = self.small_thresh * (1.0 - self.scale_border_ratio)
        s_high = self.small_thresh * (1.0 + self.scale_border_ratio)

        m_low = self.medium_thresh * (1.0 - self.scale_border_ratio)
        m_high = self.medium_thresh * (1.0 + self.scale_border_ratio)

        if xs_low <= area <= xs_high:
            if 0 not in candidates:
                candidates.append(0)
            if 1 not in candidates:
                candidates.append(1)

        if s_low <= area <= s_high:
            if 1 not in candidates:
                candidates.append(1)
            if 2 not in candidates:
                candidates.append(2)

        if m_low <= area <= m_high:
            if 2 not in candidates:
                candidates.append(2)
            if 3 not in candidates:
                candidates.append(3)

        ordered = [primary]
        for s in [0, 1, 2, 3]:
            if s != primary and s in candidates:
                ordered.append(s)
        return ordered

    def collision_priority(self, gx, gy, gi, gj, bw, bh):
        cell_cx = gi + 0.5
        cell_cy = gj + 0.5

        dist2 = (gx - cell_cx) ** 2 + (gy - cell_cy) ** 2
        area = float(bw * bh)

        size_bonus = self.small_object_bonus * (
            1.0 - min(area / max(self.medium_thresh, 1e-6), 1.0)
        )

        return float(-dist2 + size_bonus)

    def build_targets(self, outputs, targets, device):
        batch_size = outputs[0].shape[0]
        target_maps = []
        priority_maps = []

        for out in outputs:
            _, _, h, w = out.shape
            t = torch.zeros((batch_size, h, w, 5 + self.num_classes), device=device)
            p = torch.full((batch_size, h, w), fill_value=-1e9, device=device)
            target_maps.append(t)
            priority_maps.append(p)

        for b in range(batch_size):
            boxes = targets[b]["boxes"].to(device)
            labels = targets[b]["labels"].to(device)

            if boxes.numel() == 0:
                continue

            areas = boxes[:, 2] * boxes[:, 3]
            order = torch.argsort(areas, descending=False)

            for idx in order:
                cx, cy, bw, bh = boxes[idx]
                cls_id = int(labels[idx].item())

                assigned = False
                scales = self.candidate_scales(float(bw), float(bh))

                for scale_idx in scales:
                    _, _, H, W = outputs[scale_idx].shape

                    gx = float(cx * W)
                    gy = float(cy * H)

                    gi = min(W - 1, max(0, int(gx)))
                    gj = min(H - 1, max(0, int(gy)))

                    tx = gx - gi
                    ty = gy - gj
                    tw = float(bw)
                    th = float(bh)

                    # Clamp instead of discard — boundary-center objects
                    # (e.g. large buses) were silently dropped before
                    tx = max(0.0, min(tx, 0.9999))
                    ty = max(0.0, min(ty, 0.9999))

                    new_priority = self.collision_priority(gx, gy, gi, gj, tw, th)
                    old_priority = float(priority_maps[scale_idx][b, gj, gi].item())

                    if new_priority > old_priority:
                        target_maps[scale_idx][b, gj, gi, 0] = tx
                        target_maps[scale_idx][b, gj, gi, 1] = ty
                        target_maps[scale_idx][b, gj, gi, 2] = tw
                        target_maps[scale_idx][b, gj, gi, 3] = th
                        target_maps[scale_idx][b, gj, gi, 4] = 1.0
                        target_maps[scale_idx][b, gj, gi, 5:] = 0.0
                        target_maps[scale_idx][b, gj, gi, 5 + cls_id] = 1.0
                        priority_maps[scale_idx][b, gj, gi] = new_priority
                        assigned = True
                        break

                if not assigned:
                    pass

        return target_maps

In [ ]:
%%writefile /kaggle/working/YOLO_Project/training/loss.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def focal_bce_with_logits(logits, targets, alpha=0.25, gamma=2.0):
    bce              = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    probs            = torch.sigmoid(logits)
    p_t              = targets * probs + (1.0 - targets) * (1.0 - probs)
    alpha_factor     = targets * alpha + (1.0 - targets) * (1.0 - alpha)
    modulating_factor = (1.0 - p_t) ** gamma
    return alpha_factor * modulating_factor * bce


def xywh_to_xyxy(xywh):
    x, y, w, h = xywh.unbind(-1)
    return torch.stack([x - w / 2, y - h / 2, x + w / 2, y + h / 2], dim=-1)


def ciou_xyxy(box1, box2, eps=1e-7):
    """
    Complete IoU (CIoU) per Zheng et al. 2020, "Distance-IoU Loss".
    Matches ultralytics bbox_iou(ciou=True) in ultralytics/utils/metrics.py.
    Returns the CIoU score (higher = better); loss = 1 - ciou_xyxy(...).

    Three penalty terms beyond standard IoU:
      1. rho2/c2  : penalises centre-point distance between boxes
      2. alpha*v  : penalises aspect-ratio inconsistency
    where:
      rho2 = squared Euclidean distance between box centres
      c2   = squared diagonal of the smallest enclosing box
      v    = (4/pi^2) * (arctan(w_gt/h_gt) - arctan(w_pred/h_pred))^2
      alpha = v / (1 - iou + v + eps)   [weights v relative to iou gap]
    """
    # ── intersection ─────────────────────────────────────────────────────
    ix1   = torch.max(box1[..., 0], box2[..., 0])
    iy1   = torch.max(box1[..., 1], box2[..., 1])
    ix2   = torch.min(box1[..., 2], box2[..., 2])
    iy2   = torch.min(box1[..., 3], box2[..., 3])
    inter = (ix2 - ix1).clamp(min=0) * (iy2 - iy1).clamp(min=0)

    # ── areas and IoU ─────────────────────────────────────────────────────
    w1    = (box1[..., 2] - box1[..., 0]).clamp(min=0)
    h1    = (box1[..., 3] - box1[..., 1]).clamp(min=0)
    w2    = (box2[..., 2] - box2[..., 0]).clamp(min=0)
    h2    = (box2[..., 3] - box2[..., 1]).clamp(min=0)
    union = w1 * h1 + w2 * h2 - inter + eps
    iou   = inter / union

    # ── centre distance term: rho^2 / c^2 ────────────────────────────────
    cx1   = (box1[..., 0] + box1[..., 2]) / 2
    cy1   = (box1[..., 1] + box1[..., 3]) / 2
    cx2   = (box2[..., 0] + box2[..., 2]) / 2
    cy2   = (box2[..., 1] + box2[..., 3]) / 2
    rho2  = (cx1 - cx2) ** 2 + (cy1 - cy2) ** 2

    # diagonal of the smallest enclosing box, squared
    ex1   = torch.min(box1[..., 0], box2[..., 0])
    ey1   = torch.min(box1[..., 1], box2[..., 1])
    ex2   = torch.max(box1[..., 2], box2[..., 2])
    ey2   = torch.max(box1[..., 3], box2[..., 3])
    c2    = (ex2 - ex1) ** 2 + (ey2 - ey1) ** 2 + eps

    # ── aspect-ratio consistency term ────────────────────────────────────
    v     = (4.0 / (math.pi ** 2)) * (
        torch.atan(w2 / (h2 + eps)) - torch.atan(w1 / (h1 + eps))
    ) ** 2
    with torch.no_grad():
        alpha = v / (1.0 - iou + v + eps)

    return iou - rho2 / c2 - alpha * v


class YOLOStyleLoss(nn.Module):
    def __init__(
        self,
        num_classes,
        lambda_box=5.0,
        lambda_obj=1.0,
        lambda_noobj=3.0,   # raised from 2.0 — stronger background suppression
        lambda_cls=1.0,
        focal_alpha=0.25,
        focal_gamma=2.0,
        cls_weights=None,
    ):
        super().__init__()
        self.num_classes  = num_classes
        self.lambda_box   = lambda_box
        self.lambda_obj   = lambda_obj
        self.lambda_noobj = lambda_noobj
        self.lambda_cls   = lambda_cls
        self.focal_alpha  = focal_alpha
        self.focal_gamma  = focal_gamma

        if cls_weights is not None:
            w = torch.tensor(cls_weights, dtype=torch.float32)
            self.register_buffer("cls_weights", w / w.sum() * num_classes)
        else:
            self.register_buffer("cls_weights", torch.ones(num_classes))

        self.bce      = nn.BCEWithLogitsLoss(reduction="none")
        self.smoothl1 = nn.SmoothL1Loss(reduction="none")

    def _make_grid(self, H, W, device):
        """Build (gx, gy) grid tensors once and reuse for pred and tgt decoding."""
        gy, gx = torch.meshgrid(
            torch.arange(H, device=device),
            torch.arange(W, device=device),
            indexing="ij"
        )
        return gx.view(1, H, W, 1).float(), gy.view(1, H, W, 1).float()

    def decode_boxes(self, pred_xy, pred_wh, gx, gy, H, W):
        """Decode sigmoid outputs to normalised [cx, cy, w, h] using prebuilt grid."""
        cx = (pred_xy[..., 0:1] + gx) / W
        cy = (pred_xy[..., 1:2] + gy) / H
        return torch.cat([cx, cy, pred_wh[..., 0:1], pred_wh[..., 1:2]], dim=-1)

    def forward(self, outputs, target_maps):
        total_loss     = 0.0
        loss_box_total = 0.0
        loss_obj_total = 0.0
        loss_cls_total = 0.0

        for out, target in zip(outputs, target_maps):
            pred   = out.permute(0, 2, 3, 1).contiguous()
            device = pred.device
            B, H, W, C = pred.shape

            pred_xy       = torch.sigmoid(pred[..., 0:2])
            pred_wh       = torch.sigmoid(pred[..., 2:4])
            pred_obj_logit = pred[..., 4]
            pred_cls_logit = pred[..., 5:]

            tgt_xy  = target[..., 0:2]
            tgt_wh  = target[..., 2:4]
            tgt_obj = target[..., 4]
            tgt_cls = target[..., 5:]

            pos_mask = tgt_obj > 0
            neg_mask = ~pos_mask

            # ── objectness loss (focal BCE) ────────────────────────────────
            obj_all = focal_bce_with_logits(
                pred_obj_logit, tgt_obj,
                alpha=self.focal_alpha, gamma=self.focal_gamma
            )
            obj_pos = obj_all[pos_mask].mean() if pos_mask.any() else pred.new_tensor(0.0)
            obj_neg = obj_all[neg_mask].mean() if neg_mask.any() else pred.new_tensor(0.0)
            obj_loss = self.lambda_obj * obj_pos + self.lambda_noobj * obj_neg

            # ── classification loss on positives ──────────────────────────
            if pos_mask.any():
                gt_cls_idx = tgt_cls[pos_mask].argmax(dim=-1)
                sample_w   = self.cls_weights.to(device)[gt_cls_idx]
                raw_cls    = self.bce(
                    pred_cls_logit[pos_mask], tgt_cls[pos_mask]
                ).mean(dim=-1)
                cls_loss = (raw_cls * sample_w).mean()
            else:
                cls_loss = pred.new_tensor(0.0)

            # ── box loss on positives (CIoU + xy SmoothL1) ────────────────
            if pos_mask.any():
                # Build grid once, reuse for both pred and tgt
                gx, gy = self._make_grid(H, W, device)

                # xy offset SmoothL1 (stabilises early training)
                xy_loss = self.smoothl1(pred_xy[pos_mask], tgt_xy[pos_mask]).mean()

                # CIoU on decoded normalised boxes
                pred_boxes = self.decode_boxes(pred_xy, pred_wh, gx, gy, H, W)[pos_mask]
                tgt_boxes  = self.decode_boxes(tgt_xy,  tgt_wh,  gx, gy, H, W)[pos_mask]
                ciou       = ciou_xyxy(xywh_to_xyxy(pred_boxes), xywh_to_xyxy(tgt_boxes))
                ciou_loss  = (1.0 - ciou).mean()

                box_loss = xy_loss + ciou_loss
            else:
                box_loss = pred.new_tensor(0.0)

            scale_loss = self.lambda_box * box_loss + obj_loss + self.lambda_cls * cls_loss

            total_loss     = total_loss     + scale_loss
            loss_box_total = loss_box_total + box_loss.detach()
            loss_obj_total = loss_obj_total + obj_loss.detach()
            loss_cls_total = loss_cls_total + cls_loss.detach()

        return {
            "loss_total": total_loss,
            "loss_box":   loss_box_total,
            "loss_obj":   loss_obj_total,
            "loss_cls":   loss_cls_total,
        }


In [ ]:
%%writefile /kaggle/working/YOLO_Project/training/metrics.py
import time
import torch
import numpy as np
from torchvision.ops import nms

def xywh_to_xyxy_np(boxes):
    if len(boxes) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    x = boxes[:, 0]
    y = boxes[:, 1]
    w = boxes[:, 2]
    h = boxes[:, 3]
    x1 = x - w / 2
    y1 = y - h / 2
    x2 = x + w / 2
    y2 = y + h / 2
    return np.stack([x1, y1, x2, y2], axis=1)

def box_iou_np(boxes1, boxes2, eps=1e-9):
    if len(boxes1) == 0 or len(boxes2) == 0:
        return np.zeros((len(boxes1), len(boxes2)), dtype=np.float32)

    area1 = np.clip(boxes1[:, 2] - boxes1[:, 0], 0, None) * np.clip(boxes1[:, 3] - boxes1[:, 1], 0, None)
    area2 = np.clip(boxes2[:, 2] - boxes2[:, 0], 0, None) * np.clip(boxes2[:, 3] - boxes2[:, 1], 0, None)

    inter_x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])
    inter_y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])
    inter_x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])
    inter_y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])

    inter = np.clip(inter_x2 - inter_x1, 0, None) * np.clip(inter_y2 - inter_y1, 0, None)
    union = area1[:, None] + area2[None, :] - inter + eps
    return inter / union

def safe_f1(precision, recall, eps=1e-9):
    return float((2.0 * precision * recall) / (precision + recall + eps))

def decode_predictions(outputs, conf_thresh=0.45, num_classes=4, topk_total=300, nms_iou=0.40, min_wh=0.0):
    batch_size = outputs[0].shape[0]
    results = []

    for b in range(batch_size):
        all_boxes = []
        all_scores = []
        all_labels = []

        for out in outputs:
            pred = out[b].permute(1, 2, 0).contiguous()
            H, W, _ = pred.shape

            pred_xy = torch.sigmoid(pred[..., 0:2])
            pred_wh = torch.sigmoid(pred[..., 2:4])
            pred_obj = torch.sigmoid(pred[..., 4])
            pred_cls = torch.sigmoid(pred[..., 5:])

            gy, gx = torch.meshgrid(
                torch.arange(H, device=pred.device),
                torch.arange(W, device=pred.device),
                indexing="ij"
            )
            gx = gx.float()
            gy = gy.float()

            cx = (pred_xy[..., 0] + gx) / W
            cy = (pred_xy[..., 1] + gy) / H
            bw = pred_wh[..., 0]
            bh = pred_wh[..., 1]

            cls_scores, cls_ids = pred_cls.max(dim=-1)
            scores = pred_obj * cls_scores

            mask = scores > conf_thresh
            if mask.sum() == 0:
                continue

            cx = cx[mask]
            cy = cy[mask]
            bw = bw[mask]
            bh = bh[mask]
            scores_kept = scores[mask]
            labels_kept = cls_ids[mask]

            x1 = (cx - bw / 2).clamp(0, 1)
            y1 = (cy - bh / 2).clamp(0, 1)
            x2 = (cx + bw / 2).clamp(0, 1)
            y2 = (cy + bh / 2).clamp(0, 1)

            valid = (x2 > x1) & (y2 > y1)
            if min_wh > 0:
                valid = valid & ((x2 - x1) >= min_wh) & ((y2 - y1) >= min_wh)
            if valid.sum() == 0:
                continue

            all_boxes.append(torch.stack([x1, y1, x2, y2], dim=1)[valid])
            all_scores.append(scores_kept[valid])
            all_labels.append(labels_kept[valid])

        if len(all_boxes) == 0:
            results.append({
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "scores": torch.zeros((0,), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.long)
            })
            continue

        boxes = torch.cat(all_boxes, dim=0)
        scores = torch.cat(all_scores, dim=0)
        labels = torch.cat(all_labels, dim=0)

        if boxes.shape[0] > topk_total:
            topk_idx = torch.topk(scores, k=topk_total).indices
            boxes = boxes[topk_idx]
            scores = scores[topk_idx]
            labels = labels[topk_idx]

        if boxes.shape[0] > 1:
            keep_global = nms(boxes, scores, iou_threshold=0.60)
            boxes = boxes[keep_global]
            scores = scores[keep_global]
            labels = labels[keep_global]

        final_keep = []
        for cls_id in labels.unique():
            cls_mask = labels == cls_id
            cls_idx = torch.where(cls_mask)[0]
            keep = nms(boxes[cls_mask], scores[cls_mask], nms_iou)
            final_keep.append(cls_idx[keep])

        if len(final_keep) > 0:
            final_keep = torch.cat(final_keep)
            boxes = boxes[final_keep]
            scores = scores[final_keep]
            labels = labels[final_keep]
        else:
            boxes = torch.zeros((0, 4), device=boxes.device)
            scores = torch.zeros((0,), device=scores.device)
            labels = torch.zeros((0,), dtype=torch.long, device=labels.device)

        results.append({
            "boxes": boxes.detach().cpu(),
            "scores": scores.detach().cpu(),
            "labels": labels.detach().cpu()
        })

    return results

def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))

    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])
    return float(ap)

def _build_class_data(all_preds, all_targets, cls_id):
    preds_cls = []
    gt_boxes_by_img = {}
    gt_count = 0

    for img_idx, target in enumerate(all_targets):
        gt_boxes = target["boxes"].cpu().numpy()
        gt_labels = target["labels"].cpu().numpy()
        mask = gt_labels == cls_id
        gt_boxes = xywh_to_xyxy_np(gt_boxes[mask])
        gt_boxes_by_img[img_idx] = gt_boxes
        gt_count += len(gt_boxes)

    for img_idx, pred in enumerate(all_preds):
        pred_boxes = pred["boxes"].cpu().numpy()
        pred_scores = pred["scores"].cpu().numpy()
        pred_labels = pred["labels"].cpu().numpy()

        mask = pred_labels == cls_id
        for box, score in zip(pred_boxes[mask], pred_scores[mask]):
            preds_cls.append((img_idx, float(score), box))

    preds_cls.sort(key=lambda x: x[1], reverse=True)
    return preds_cls, gt_boxes_by_img, gt_count

def _evaluate_class_at_iou(preds_cls, gt_boxes_by_img, gt_count, iou_thresh):
    matched = {
        img_idx: np.zeros(len(boxes), dtype=bool)
        for img_idx, boxes in gt_boxes_by_img.items()
    }

    tp = np.zeros(len(preds_cls), dtype=np.float32)
    fp = np.zeros(len(preds_cls), dtype=np.float32)

    for i, (img_idx, score, pred_box) in enumerate(preds_cls):
        gt_boxes = gt_boxes_by_img[img_idx]

        if len(gt_boxes) == 0:
            fp[i] = 1
            continue

        ious = box_iou_np(np.array([pred_box]), gt_boxes)[0]
        best_idx = int(np.argmax(ious))
        best_iou = float(ious[best_idx])

        if best_iou >= iou_thresh and not matched[img_idx][best_idx]:
            tp[i] = 1
            matched[img_idx][best_idx] = True
        else:
            fp[i] = 1

    cum_tp = np.cumsum(tp)
    cum_fp = np.cumsum(fp)

    recall = cum_tp / max(gt_count, 1)
    precision = cum_tp / np.maximum(cum_tp + cum_fp, 1e-9)

    ap = compute_ap(recall, precision) if gt_count > 0 else 0.0

    TP = int(tp.sum())
    FP = int(fp.sum())
    FN = int(gt_count - TP)

    precision_scalar = float(TP / max(TP + FP, 1))
    recall_scalar = float(TP / max(TP + FN, 1))
    f1_scalar = safe_f1(precision_scalar, recall_scalar)

    return {
        "AP": float(ap),
        "Precision": precision_scalar,
        "Recall": recall_scalar,
        "F1": f1_scalar,
        "GT": int(gt_count),
        "GT Count": int(gt_count),
        "TP": TP,
        "FP": FP,
        "FN": FN
    }

def evaluate_map(all_preds, all_targets, num_classes=4, iou_thresh=0.5, iou_thresholds=None):
    if iou_thresholds is None:
        iou_thresholds = np.arange(0.50, 0.96, 0.05, dtype=np.float32)

    per_class = {}
    aps_50 = []
    aps_5095 = []

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for cls_id in range(num_classes):
        preds_cls, gt_boxes_by_img, gt_count = _build_class_data(all_preds, all_targets, cls_id)

        stats_50 = _evaluate_class_at_iou(preds_cls, gt_boxes_by_img, gt_count, iou_thresh)
        per_class[cls_id] = {
            **stats_50,
            "AP@0.5": stats_50["AP"],
        }

        aps_50.append(stats_50["AP"])
        total_tp += stats_50["TP"]
        total_fp += stats_50["FP"]
        total_fn += stats_50["FN"]

        cls_ap_thresholds = []
        for thr in iou_thresholds:
            stats_thr = _evaluate_class_at_iou(preds_cls, gt_boxes_by_img, gt_count, float(thr))
            cls_ap_thresholds.append(stats_thr["AP"])
        aps_5095.extend(cls_ap_thresholds)

    mAP50 = float(np.mean(aps_50)) if aps_50 else 0.0
    mAP5095 = float(np.mean(aps_5095)) if aps_5095 else 0.0
    precision_all = float(total_tp / max(total_tp + total_fp, 1))
    recall_all = float(total_tp / max(total_tp + total_fn, 1))
    f1_all = safe_f1(precision_all, recall_all)

    return {
        "mAP@0.5": mAP50,
        "mAP@0.5:0.95": mAP5095,
        "Precision": precision_all,
        "Recall": recall_all,
        "F1": f1_all,
        "per_class": per_class
    }

@torch.no_grad()
def benchmark_dataloader_fps(
    model,
    loader,
    device,
    num_classes,
    conf_thresh=0.45,
    nms_iou=0.40,
    max_batches=10,
    warmup_batches=2,
    include_decode=True
):
    model.eval()

    batches = []
    for batch_idx, (images, _, _) in enumerate(loader):
        batches.append(images.to(device))
        if batch_idx + 1 >= max_batches:
            break

    if len(batches) == 0:
        return {"FPS": 0.0, "ms_per_image": 0.0}

    for i in range(min(warmup_batches, len(batches))):
        outputs = model(batches[i])
        if include_decode:
            _ = decode_predictions(
                outputs,
                conf_thresh=conf_thresh,
                num_classes=num_classes,
                nms_iou=nms_iou
            )

    if device.type == "cuda":
        torch.cuda.synchronize()

    total_images = 0
    start = time.time()

    for images in batches:
        outputs = model(images)
        if include_decode:
            _ = decode_predictions(
                outputs,
                conf_thresh=conf_thresh,
                num_classes=num_classes,
                nms_iou=nms_iou
            )
        total_images += images.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.time() - start
    fps = total_images / max(elapsed, 1e-9)
    ms_per_image = (elapsed / max(total_images, 1)) * 1000.0
    return {"FPS": float(fps), "ms_per_image": float(ms_per_image)}


**Part 5: Add project root to path and import**

In [ ]:
import sys, os, importlib
sys.path.append(PROJECT_ROOT)

# In Kaggle sessions, these modules may already be cached from an earlier notebook run.
# Clear them so the rewritten ResNet34 files are imported instead of stale ResNet18 code.
for _mod in [
    "models.backbone_resnet",
    "models.yolo11s",
    "data.transforms",
    "data.dataset",
    "training.assigner",
    "training.loss",
    "training.metrics",
]:
    if _mod in sys.modules:
        del sys.modules[_mod]
importlib.invalidate_caches()

from models.yolo11s import YOLO11sResNet34
from data.dataset import YOLODetectionDataset, collate_fn
from data.transforms import get_train_transforms, get_test_transforms, IMAGENET_MEAN, IMAGENET_STD
from training.assigner import MultiScaleAssigner
from training.loss import YOLOStyleLoss
from training.metrics import decode_predictions, evaluate_map, benchmark_dataloader_fps

# ── Deferred checkpoint load (set LOAD_PRETRAINED_RUN in Cell 2) ────────────
if LOAD_PRETRAINED_RUN is not None:
    if not os.path.exists(LOAD_PRETRAINED_RUN):
        raise FileNotFoundError(f"Checkpoint not found: {LOAD_PRETRAINED_RUN}")

    model = YOLO11sResNet34(
        num_classes=NUM_CLASSES,
        pretrained_backbone=False
    ).to(device)

    ckpt = torch.load(LOAD_PRETRAINED_RUN, map_location=device)
    try:
        model.load_state_dict(ckpt["model_state"])
    except RuntimeError as e:
        raise RuntimeError(
            "Checkpoint architecture mismatch. This notebook now uses YOLO11sResNet34. "
            "Use a matching ResNet34 checkpoint or set LOAD_PRETRAINED_RUN = None."
        ) from e
    model.eval()

    print(f"Loaded checkpoint: {LOAD_PRETRAINED_RUN}")
    print(f"  Epoch: {ckpt['epoch']}")
    print(f"  Metrics: {ckpt['metrics']}")
else:
    print("No checkpoint specified — will train from scratch.")


**Part 6: Build datasets and dataloaders**

In [ ]:
from data.dataset import extract_image_number

all_train_images = list_images(TRAIN_IMG_DIR)

val_images = []
train_split_images = []

for name in all_train_images:
    img_num = extract_image_number(name)
    if img_num is not None and img_num % 10 == 0:
        val_images.append(name)
    else:
        train_split_images.append(name)

# Fallback in case filenames do not end with digits
if len(val_images) == 0:
    val_images = all_train_images[::10]
    train_split_images = [n for n in all_train_images if n not in set(val_images)]

assert len(train_split_images) > 0, "No training images after split."
assert len(val_images) > 0, "No validation images after split."
assert set(train_split_images).isdisjoint(set(val_images)), "Train/val overlap detected."

train_dataset = YOLODetectionDataset(
    TRAIN_IMG_DIR, TRAIN_LBL_DIR,
    img_size=IMG_SIZE,
    transforms=get_train_transforms(),
    image_names=train_split_images,
    use_letterbox=False,
    augment=False
)

val_dataset = YOLODetectionDataset(
    TRAIN_IMG_DIR, TRAIN_LBL_DIR,
    img_size=IMG_SIZE,
    transforms=get_test_transforms(),
    image_names=val_images,
    use_letterbox=False
)

test_dataset = YOLODetectionDataset(
    TEST_IMG_DIR, TEST_LBL_DIR,
    img_size=IMG_SIZE,
    transforms=get_test_transforms(),
    use_letterbox=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
    persistent_workers=True,
    collate_fn=collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
    persistent_workers=True,
    collate_fn=collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
    persistent_workers=True,
    collate_fn=collate_fn,
    worker_init_fn=seed_worker,
    generator=loader_generator
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))
print("Train images :", len(train_dataset))
print("Val images   :", len(val_dataset))
print("Test images  :", len(test_dataset))


**Part 7: Visualise training samples with GT boxes**

In [ ]:
def denormalize_img_gt(img_tensor, mean, std):
    mean = torch.tensor(mean, device=img_tensor.device).view(3, 1, 1)
    std = torch.tensor(std, device=img_tensor.device).view(3, 1, 1)
    img = img_tensor * std + mean
    return img.clamp(0, 1)

def draw_yolo_boxes(img_tensor, target, class_names):
    img_tensor = denormalize_img_gt(
        img_tensor,
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = (img * 255).astype(np.uint8).copy()
    h, w = img.shape[:2]

    boxes = target["boxes"].cpu().numpy()
    labels = target["labels"].cpu().numpy()

    for box, cls_id in zip(boxes, labels):
        cx, cy, bw, bh = box
        x1 = int((cx - bw / 2) * w)
        y1 = int((cy - bh / 2) * h)
        x2 = int((cx + bw / 2) * w)
        y2 = int((cy + bh / 2) * h)

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(
            img,
            class_names[int(cls_id)],
            (x1, max(0, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 0, 0),
            2
        )

    return img

images, targets, names = next(iter(train_loader))
plt.figure(figsize=(15, 10))
for i in range(min(4, len(images))):
    plt.subplot(2, 2, i + 1)
    vis = draw_yolo_boxes(images[i], targets[i], CLASS_NAMES)
    plt.imshow(vis)
    plt.title(names[i])
    plt.axis("off")
plt.tight_layout()
plt.show()

**Part 8: Dataset audit**

In [ ]:
def count_class_distribution(lbl_dir, image_names=None):
    counter = Counter()
    if image_names is None:
        label_files = [f for f in os.listdir(lbl_dir) if f.endswith(".txt")]
    else:
        label_files = [os.path.splitext(n)[0] + ".txt" for n in image_names]

    for f in label_files:
        path = os.path.join(lbl_dir, f)
        if not os.path.exists(path):
            continue
        with open(path, "r") as fp:
            for line in fp:
                vals = line.strip().split()
                if len(vals) == 5:
                    cls_id = int(float(vals[0]))
                    counter[cls_id] += 1
    return counter

train_counter = count_class_distribution(TRAIN_LBL_DIR, train_split_images)
val_counter = count_class_distribution(TRAIN_LBL_DIR, val_images)
test_counter = count_class_distribution(TEST_LBL_DIR)

print("Train class distribution:")
for k, v in sorted(train_counter.items()):
    print(f"  {k} ({CLASS_NAMES.get(k, k)}): {v}")

print("\nVal class distribution:")
for k, v in sorted(val_counter.items()):
    print(f"  {k} ({CLASS_NAMES.get(k, k)}): {v}")

print("\nTest class distribution:")
for k, v in sorted(test_counter.items()):
    print(f"  {k} ({CLASS_NAMES.get(k, k)}): {v}")

total_train_instances = sum(train_counter.values())

# Inverse-frequency class weights (sqrt-softened, capped at 2.5)
# Cap lowered from 5.0: Motorcycle at 2.47 caused 447 FP (19x ratio)
# A tighter cap preserves minority-class boost without overcorrecting
import math as _math
_counts = [train_counter.get(i, 1) for i in range(NUM_CLASSES)]
_total  = sum(_counts)
CLASS_WEIGHTS = [
    round(min(_math.sqrt(_total / (NUM_CLASSES * c)), 2.5), 4)
    for c in _counts
]

print("\nClass weights (inverse-frequency, sqrt-softened, cap=2.5):")
for c, w in enumerate(CLASS_WEIGHTS):
    print(f"  {CLASS_NAMES[c]}: {w:.4f}")


**Part 9 : Build model**

In [ ]:
# Only build the model fresh if no checkpoint was loaded in Cell 21
if LOAD_PRETRAINED_RUN is None:
    model = YOLO11sResNet34(
        num_classes=NUM_CLASSES,
        pretrained_backbone=True
    ).to(device)

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    outputs = model(dummy)

for i, out in enumerate(outputs):
    print(f"Scale {i}: shape = {out.shape}")


**Part 10: Optimiser, scheduler, assigner, loss**

In [ ]:
def freeze_backbone_for_warmup(model):
    for p in model.backbone.parameters():
        p.requires_grad = False

    for p in model.backbone.layer4.parameters():
        p.requires_grad = True

def unfreeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = True

freeze_backbone_for_warmup(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")

In [ ]:
optimizer = torch.optim.AdamW([
    {
        "params": filter(lambda p: p.requires_grad, model.backbone.parameters()),
        "lr": 1e-4
    },
    {
        "params": list(model.sppf.parameters()) +
                  list(model.attn.parameters()) +
                  list(model.neck_p4.parameters()) +
                  list(model.neck_p3.parameters()) +
                  list(model.neck_p2.parameters()) +
                  list(model.down_p3.parameters()) +
                  list(model.neck_p3_out.parameters()) +
                  list(model.down_p4.parameters()) +
                  list(model.neck_p4_out.parameters()) +
                  list(model.down_p5.parameters()) +
                  list(model.neck_p5_out.parameters()) +
                  list(model.detect.parameters()),
        "lr": 1e-3
    }
], weight_decay=1e-4)

# Warmup-phase scheduler (epochs 0-4, backbone frozen)
# At unfreeze (epoch 5), Cell 36 replaces this with T_max=45 for the main phase
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,
    eta_min=1e-4
)

assigner = MultiScaleAssigner(
    num_classes=NUM_CLASSES,
    img_size=IMG_SIZE,
    strides=(4, 8, 16, 32),
    xsmall_thresh=0.015,
    small_thresh=0.07,     # was 0.05 — matches assigner.py update
    medium_thresh=0.25,    # was 0.18 — routes buses to P4 not just P5
    scale_border_ratio=0.20,
    small_object_bonus=0.20
)

criterion = YOLOStyleLoss(
    num_classes=NUM_CLASSES,
    lambda_box=5.0,
    lambda_obj=1.0,
    lambda_noobj=3.0,
    lambda_cls=1.0,
    focal_alpha=0.25,
    focal_gamma=2.0,
    cls_weights=CLASS_WEIGHTS,
)

**Part 11: Training and validation function**

In [ ]:
def train_one_epoch(model, loader, optimizer, assigner, criterion, device):
    model.train()
    running = defaultdict(float)

    for images, targets, names in loader:
        images = images.to(device)

        outputs = model(images)
        target_maps = assigner.build_targets(outputs, targets, device=device)
        loss_dict = criterion(outputs, target_maps)
        loss = loss_dict["loss_total"]

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        running["loss_total"] += float(loss_dict["loss_total"].item())
        running["loss_box"] += float(loss_dict["loss_box"].item())
        running["loss_obj"] += float(loss_dict["loss_obj"].item())
        running["loss_cls"] += float(loss_dict["loss_cls"].item())

    for k in running:
        running[k] /= len(loader)
    return dict(running)

@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    assigner,
    criterion,
    device,
    num_classes,
    conf_thresh=None,
    nms_iou=None,
    ap_conf_thresh=0.01,
    measure_fps=False
):
    model.eval()
    running = defaultdict(float)

    conf_thresh = EVAL_CONF_THRESH if conf_thresh is None else conf_thresh
    nms_iou = EVAL_NMS_IOU if nms_iou is None else nms_iou

    all_preds_ap = []
    all_preds_report = []
    all_targets = []

    for images, targets, names in loader:
        images = images.to(device)

        outputs = model(images)
        target_maps = assigner.build_targets(outputs, targets, device=device)
        loss_dict = criterion(outputs, target_maps)

        running["loss_total"] += float(loss_dict["loss_total"].item())
        running["loss_box"] += float(loss_dict["loss_box"].item())
        running["loss_obj"] += float(loss_dict["loss_obj"].item())
        running["loss_cls"] += float(loss_dict["loss_cls"].item())

        preds_ap = decode_predictions(
            outputs,
            conf_thresh=ap_conf_thresh,
            nms_iou=nms_iou,
            num_classes=num_classes
        )
        preds_report = decode_predictions(
            outputs,
            conf_thresh=conf_thresh,
            nms_iou=nms_iou,
            num_classes=num_classes
        )

        all_preds_ap.extend(preds_ap)
        all_preds_report.extend(preds_report)
        all_targets.extend(targets)

    for k in running:
        running[k] /= len(loader)

    metrics_ap = evaluate_map(all_preds_ap, all_targets, num_classes=num_classes, iou_thresh=0.5)
    metrics_report = evaluate_map(all_preds_report, all_targets, num_classes=num_classes, iou_thresh=0.5)

    metrics = {
        "mAP@0.5": metrics_ap["mAP@0.5"],
        "mAP@0.5:0.95": metrics_ap["mAP@0.5:0.95"],
        "Precision": metrics_report["Precision"],
        "Recall": metrics_report["Recall"],
        "F1": metrics_report["F1"],
        "per_class": metrics_report["per_class"],
    }

    if measure_fps:
        fps_stats = benchmark_dataloader_fps(
            model=model,
            loader=loader,
            device=device,
            num_classes=num_classes,
            conf_thresh=conf_thresh,
            nms_iou=nms_iou,
            max_batches=min(10, len(loader)),
            warmup_batches=min(2, len(loader)),
            include_decode=True,
        )
        metrics["FPS"] = fps_stats["FPS"]
        metrics["ms_per_image"] = fps_stats["ms_per_image"]
    else:
        metrics["FPS"] = 0.0
        metrics["ms_per_image"] = 0.0

    return dict(running), metrics


**Part 12: Training**

In [ ]:
import time
from datetime import datetime

# ── Run timestamp — generated once per notebook execution ───────────────────
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

num_epochs = 100  # extended from 75; train loss still falling at ep75
warmup_freeze_epochs = 5
best_map = -1.0
best_map_epoch = -1
best_map_pfloor = -1.0
best_map_pfloor_epoch = -1
MIN_PRECISION_FOR_MAP_CKPT = 0.27
history = []

# ── Timestamped output directories for this run ─────────────────────────────
RUN_DIR = f"{PROJECT_ROOT}/outputs/runs/{RUN_TIMESTAMP}"
CKPT_DIR = f"{RUN_DIR}/checkpoints"
LOG_DIR = f"{RUN_DIR}/logs"
PRED_DIR = f"{RUN_DIR}/predictions"
BLIND_DIR = f"{RUN_DIR}/blind_test_analysis"

for _d in [CKPT_DIR, LOG_DIR, PRED_DIR, BLIND_DIR]:
    os.makedirs(_d, exist_ok=True)

print(f"Run ID : {RUN_TIMESTAMP}", flush=True)
print(f"Run dir: {RUN_DIR}", flush=True)
print(f"mAP precision floor: {MIN_PRECISION_FOR_MAP_CKPT:.2f}", flush=True)

best_map_path = f"{CKPT_DIR}/best_map_{RUN_TIMESTAMP}.pth"
best_map_pfloor_path = f"{CKPT_DIR}/best_map_pfloor_{RUN_TIMESTAMP}.pth"
# best_path kept as alias → points to standard best mAP model for downstream cells
best_path = best_map_path

for epoch in range(num_epochs):
    t0 = time.time()

    if epoch == warmup_freeze_epochs:
        print("\nUnfreezing full backbone...", flush=True)
        unfreeze_backbone(model)

        optimizer = torch.optim.AdamW([
            {"params": model.backbone.parameters(), "lr": 1e-4},
            {"params": list(model.sppf.parameters()) +
                      list(model.attn.parameters()) +
                      list(model.neck_p4.parameters()) +
                      list(model.neck_p3.parameters()) +
                      list(model.neck_p2.parameters()) +
                      list(model.down_p3.parameters()) +
                      list(model.neck_p3_out.parameters()) +
                      list(model.down_p4.parameters()) +
                      list(model.neck_p4_out.parameters()) +
                      list(model.down_p5.parameters()) +
                      list(model.neck_p5_out.parameters()) +
                      list(model.detect.parameters()), "lr": 1e-3}
        ], weight_decay=1e-4)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=num_epochs - warmup_freeze_epochs,
            eta_min=1e-5
        )
        print("Optimizer reset. Starting main phase.\n", flush=True)

    train_stats = train_one_epoch(model, train_loader, optimizer, assigner, criterion, device)
    val_stats, val_metrics = validate_one_epoch(
        model,
        val_loader,
        assigner,
        criterion,
        device,
        NUM_CLASSES,
        conf_thresh=EVAL_CONF_THRESH,
        nms_iou=EVAL_NMS_IOU,
        ap_conf_thresh=0.01,
        measure_fps=True
    )

    scheduler.step()
    elapsed = time.time() - t0

    row = {
        "epoch": epoch + 1,
        "train_loss": train_stats["loss_total"],
        "train_box": train_stats["loss_box"],
        "train_obj": train_stats["loss_obj"],
        "train_cls": train_stats["loss_cls"],
        "val_loss": val_stats["loss_total"],
        "val_box": val_stats["loss_box"],
        "val_obj": val_stats["loss_obj"],
        "val_cls": val_stats["loss_cls"],
        "mAP@0.5": val_metrics["mAP@0.5"],
        "mAP@0.5:0.95": val_metrics["mAP@0.5:0.95"],
        "Precision": val_metrics["Precision"],
        "Recall": val_metrics["Recall"],
        "F1": val_metrics["F1"],
        "FPS": val_metrics["FPS"],
        "ms_per_image": val_metrics["ms_per_image"],
        "epoch_time_sec": elapsed,
    }

    _p = row["Precision"]
    _r = row["Recall"]
    _map = row["mAP@0.5"]
    _f05 = (1 + 0.25) * _p * _r / (0.25 * _p + _r + 1e-9)
    row["F0.5"] = _f05

    history.append(row)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"train={row['train_loss']:.4f} | "
        f"val={row['val_loss']:.4f} | "
        f"box={row['val_box']:.4f} | "
        f"obj={row['val_obj']:.4f} | "
        f"cls={row['val_cls']:.4f} | "
        f"mAP@0.5={row['mAP@0.5']:.4f} | "
        f"mAP@0.5:0.95={row['mAP@0.5:0.95']:.4f} | "
        f"P={row['Precision']:.4f} | "
        f"R={row['Recall']:.4f} | "
        f"F1={row['F1']:.4f} | "
        f"F0.5={row['F0.5']:.4f} | "
        f"FPS={row['FPS']:.2f} | "
        f"ms/img={row['ms_per_image']:.2f} | "
        f"{elapsed:.0f}s",
        flush=True
    )

    if _map > best_map:
        best_map = _map
        best_map_epoch = epoch + 1
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch + 1,
            "metrics": row,
            "run_timestamp": RUN_TIMESTAMP,
            "checkpoint_type": "best_map",
        }, best_map_path)
        print(
            f"  [mAP] Best={best_map:.4f} "
            f"(mAP@0.5:0.95={row['mAP@0.5:0.95']:.4f} "
            f"F1={row['F1']:.4f} F0.5={_f05:.4f} "
            f"P={_p:.4f} R={_r:.4f}) — {best_map_path.split('/')[-1]}",
            flush=True
        )

    if _map > best_map_pfloor and _p >= MIN_PRECISION_FOR_MAP_CKPT:
        best_map_pfloor = _map
        best_map_pfloor_epoch = epoch + 1
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch + 1,
            "metrics": row,
            "run_timestamp": RUN_TIMESTAMP,
            "checkpoint_type": "best_map_pfloor",
            "min_precision_for_map_ckpt": MIN_PRECISION_FOR_MAP_CKPT,
        }, best_map_pfloor_path)
        print(
            f"  [mAP+Pfloor] Best={best_map_pfloor:.4f} "
            f"(mAP@0.5:0.95={row['mAP@0.5:0.95']:.4f} "
            f"F1={row['F1']:.4f} F0.5={_f05:.4f} "
            f"P={_p:.4f} R={_r:.4f}) — {best_map_pfloor_path.split('/')[-1]}",
            flush=True
        )
    elif _map > best_map_pfloor and _p < MIN_PRECISION_FOR_MAP_CKPT:
        print(
            f"  [mAP+Pfloor] Skipped={_map:.4f} — P={_p:.4f} below floor "
            f"{MIN_PRECISION_FOR_MAP_CKPT:.2f} (current best: epoch {best_map_pfloor_epoch})",
            flush=True
        )

if best_map_pfloor_epoch == -1:
    print(
        f"WARNING: no checkpoint met the precision floor {MIN_PRECISION_FOR_MAP_CKPT:.2f}. "
        "Lower the threshold and retrain if you want the gated checkpoint.",
        flush=True
    )


**Part 13: Plot training curves**

In [ ]:
hist_df = pd.DataFrame(history)

training_history_path = f"{LOG_DIR}/training_history_{RUN_TIMESTAMP}.csv"
hist_df.to_csv(training_history_path, index=False)
print("Saved training history to:", training_history_path)

best_idx = hist_df["mAP@0.5"].idxmax()
best_row = hist_df.loc[best_idx]

summary_df = pd.DataFrame([{
    "Best Epoch": int(best_row["epoch"]),
    "Best mAP@0.5": float(best_row["mAP@0.5"]),
    "Best mAP@0.5:0.95": float(best_row["mAP@0.5:0.95"]),
    "Precision": float(best_row["Precision"]),
    "Recall": float(best_row["Recall"]),
    "F1": float(best_row["F1"]),
    "F0.5": float(best_row["F0.5"]),
    "FPS": float(best_row["FPS"]),
    "ms_per_image": float(best_row["ms_per_image"]),
    "Val Total Loss": float(best_row["val_loss"]),
    "Val Box Loss": float(best_row["val_box"]),
    "Val Obj Loss": float(best_row["val_obj"]),
    "Val Cls Loss": float(best_row["val_cls"]),
}])

summary_path = f"{LOG_DIR}/best_epoch_summary_{RUN_TIMESTAMP}.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved best-epoch summary to:", summary_path)

display(summary_df)
hist_df.tail()


In [ ]:
import os
import matplotlib.pyplot as plt

FIG_DIR = f"{RUN_DIR}/figures"
os.makedirs(FIG_DIR, exist_ok=True)

def save_plot(x, ys, labels, title, ylabel, filename):
    plt.figure(figsize=(8, 5))
    for y, label in zip(ys, labels):
        plt.plot(x, y, label=label)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, filename)
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print("Saved plot to:", out_path)
    plt.show()

save_plot(
    hist_df["epoch"],
    [hist_df["train_loss"], hist_df["val_loss"]],
    ["Train Total Loss", "Val Total Loss"],
    "Total Loss",
    "Loss",
    f"total_loss_{RUN_TIMESTAMP}.png"
)

save_plot(
    hist_df["epoch"],
    [hist_df["train_box"], hist_df["val_box"]],
    ["Train Box Loss", "Val Box Loss"],
    "Box Loss",
    "Loss",
    f"box_loss_{RUN_TIMESTAMP}.png"
)

save_plot(
    hist_df["epoch"],
    [hist_df["train_obj"], hist_df["val_obj"]],
    ["Train Obj Loss", "Val Obj Loss"],
    "Objectness Loss",
    "Loss",
    f"objectness_loss_{RUN_TIMESTAMP}.png"
)

save_plot(
    hist_df["epoch"],
    [hist_df["train_cls"], hist_df["val_cls"]],
    ["Train Cls Loss", "Val Cls Loss"],
    "Classification Loss",
    "Loss",
    f"classification_loss_{RUN_TIMESTAMP}.png"
)

metric_series = [
    hist_df["mAP@0.5"],
    hist_df["mAP@0.5:0.95"],
    hist_df["Precision"],
    hist_df["Recall"],
    hist_df["F1"],
]
metric_labels = [
    "mAP@0.5",
    "mAP@0.5:0.95",
    "Precision",
    "Recall",
    "F1",
]
if "F0.5" in hist_df.columns:
    metric_series.append(hist_df["F0.5"])
    metric_labels.append("F0.5")

save_plot(
    hist_df["epoch"],
    metric_series,
    metric_labels,
    "Detection Metrics",
    "Metric",
    f"detection_metrics_{RUN_TIMESTAMP}.png"
)

save_plot(
    hist_df["epoch"],
    [hist_df["FPS"]],
    ["FPS"],
    "Inference Speed",
    "FPS",
    f"inference_fps_{RUN_TIMESTAMP}.png"
)

save_plot(
    hist_df["epoch"],
    [hist_df["ms_per_image"]],
    ["ms/image"],
    "Inference Time per Image",
    "Milliseconds",
    f"inference_time_{RUN_TIMESTAMP}.png"
)


# Part 14: Load saved checkpoints


In [ ]:
# ── Load both checkpoints for dual evaluation ────────────────────────────────
def load_checkpoint(path, device):
    m = YOLO11sResNet34(num_classes=NUM_CLASSES, pretrained_backbone=False).to(device)
    ckpt = torch.load(path, map_location=device)
    m.load_state_dict(ckpt["model_state"])
    m.eval()
    return m, ckpt

assert os.path.exists(best_map_path), f"Missing checkpoint: {best_map_path}"
assert os.path.exists(best_map_pfloor_path), (
    f"Missing checkpoint: {best_map_pfloor_path}. "
    f"No epoch may have met MIN_PRECISION_FOR_MAP_CKPT={MIN_PRECISION_FOR_MAP_CKPT:.2f}."
)

model_map, ckpt_map = load_checkpoint(best_map_path, device)
model_map_pfloor, ckpt_map_pfloor = load_checkpoint(best_map_pfloor_path, device)

CHECKPOINT_MODELS = {
    "best_map": {
        "label": "best mAP checkpoint",
        "model": model_map,
        "ckpt": ckpt_map,
        "path": best_map_path,
    },
    "best_map_pfloor": {
        "label": f"best mAP + precision floor checkpoint (P ≥ {MIN_PRECISION_FOR_MAP_CKPT:.2f})",
        "model": model_map_pfloor,
        "ckpt": ckpt_map_pfloor,
        "path": best_map_pfloor_path,
    },
}

ARTIFACT_DIRS = {}
for tag in CHECKPOINT_MODELS:
    pred_subdir = os.path.join(PRED_DIR, tag)
    blind_subdir = os.path.join(BLIND_DIR, tag)
    os.makedirs(pred_subdir, exist_ok=True)
    os.makedirs(blind_subdir, exist_ok=True)
    ARTIFACT_DIRS[tag] = {
        "pred_dir": pred_subdir,
        "blind_dir": blind_subdir,
    }

for tag, cfg in CHECKPOINT_MODELS.items():
    print(f"── {cfg['label']} ─────────────────────────────────────────────")
    print("  Tag   :", tag)
    print("  Path  :", cfg["path"])
    print("  Epoch :", cfg["ckpt"]["epoch"])
    m = cfg["ckpt"]["metrics"]
    print(
        f"  Val   : mAP@0.5={m.get('mAP@0.5', 0):.4f}  "
        f"mAP@0.5:0.95={m.get('mAP@0.5:0.95', 0):.4f}  "
        f"P={m.get('Precision', 0):.4f}  R={m.get('Recall', 0):.4f}  "
        f"F1={m.get('F1', 0):.4f}  F0.5={m.get('F0.5', 0):.4f}"
    )
    if tag == "best_map_pfloor":
        print(f"  Floor : {MIN_PRECISION_FOR_MAP_CKPT:.2f}")
    print()

# canonical model alias for downstream compatibility
model = model_map


# Part 15: Final evaluation table for both checkpoints


In [ ]:
# ── Test evaluation: both best_map and best_map_pfloor models ───────────────
def run_test_eval(mdl, label, epoch):
    test_stats, metrics = validate_one_epoch(
        mdl,
        test_loader,
        assigner,
        criterion,
        device,
        NUM_CLASSES,
        conf_thresh=EVAL_CONF_THRESH,
        nms_iou=EVAL_NMS_IOU,
        ap_conf_thresh=0.01,
        measure_fps=True
    )

    _p = metrics["Precision"]
    _r = metrics["Recall"]
    _f1 = metrics["F1"]
    _f05 = (1 + 0.25) * _p * _r / (0.25 * _p + _r + 1e-9)

    print(f"\n{'='*72}")
    print(f"  {label}  (epoch {epoch})")
    print(f"{'='*72}")
    print(f"  mAP@0.5       : {metrics['mAP@0.5']:.4f}")
    print(f"  mAP@0.5:0.95  : {metrics['mAP@0.5:0.95']:.4f}")
    print(f"  Precision     : {_p:.4f}")
    print(f"  Recall        : {_r:.4f}")
    print(f"  F1            : {_f1:.4f}")
    print(f"  F0.5          : {_f05:.4f}")
    print(f"  FPS           : {metrics['FPS']:.2f}")
    print(f"  ms_per_image  : {metrics['ms_per_image']:.2f}")
    print(
        f"  Test losses   : total={test_stats['loss_total']:.4f} "
        f"box={test_stats['loss_box']:.4f} obj={test_stats['loss_obj']:.4f} cls={test_stats['loss_cls']:.4f}"
    )
    print(f"  {'Class':<12} {'AP50':>7} {'P':>7} {'R':>7} {'F1':>7} {'TP':>5} {'FP':>5} {'FN':>5} {'GT':>5}")
    for cid, s in metrics["per_class"].items():
        print(
            f"  {CLASS_NAMES[cid]:<12} {s.get('AP@0.5', s.get('AP', 0)):>7.3f} "
            f"{s['Precision']:>7.3f} {s['Recall']:>7.3f} {s.get('F1', 0):>7.3f} "
            f"{s['TP']:>5} {s['FP']:>5} {s['FN']:>5} {s.get('GT', s.get('GT Count', 0)):>5}"
        )

    return test_stats, metrics

test_results = {}
test_loss_results = {}
per_class_dfs = {}
test_summary_rows = []

for tag, cfg in CHECKPOINT_MODELS.items():
    test_stats, metrics = run_test_eval(cfg["model"], cfg["label"], cfg["ckpt"]["epoch"])
    test_results[tag] = metrics
    test_loss_results[tag] = test_stats

    rows = [{"Class ID": cid, "Class Name": CLASS_NAMES[cid], **s}
            for cid, s in metrics["per_class"].items()]
    per_class_dfs[tag] = pd.DataFrame(rows)

    path = f"{LOG_DIR}/per_class_metrics_{tag}_{RUN_TIMESTAMP}.csv"
    per_class_dfs[tag].to_csv(path, index=False)
    print(f"\nSaved {tag} per-class CSV: {path}")

    test_summary_rows.append({
        "model_tag": tag,
        "label": cfg["label"],
        "epoch": int(cfg["ckpt"]["epoch"]),
        "mAP@0.5": float(metrics["mAP@0.5"]),
        "mAP@0.5:0.95": float(metrics["mAP@0.5:0.95"]),
        "Precision": float(metrics["Precision"]),
        "Recall": float(metrics["Recall"]),
        "F1": float(metrics["F1"]),
        "F0.5": float((1 + 0.25) * metrics["Precision"] * metrics["Recall"] / (0.25 * metrics["Precision"] + metrics["Recall"] + 1e-9)),
        "FPS": float(metrics["FPS"]),
        "ms_per_image": float(metrics["ms_per_image"]),
        "test_loss_total": float(test_stats["loss_total"]),
        "test_loss_box": float(test_stats["loss_box"]),
        "test_loss_obj": float(test_stats["loss_obj"]),
        "test_loss_cls": float(test_stats["loss_cls"]),
    })

test_summary_df = pd.DataFrame(test_summary_rows)
test_summary_csv = os.path.join(LOG_DIR, f"test_summary_{RUN_TIMESTAMP}.csv")
test_summary_df.to_csv(test_summary_csv, index=False)
print("\nSaved overall test summary to:", test_summary_csv)
display(test_summary_df)

# canonical aliases for downstream compatibility
test_metrics = test_results["best_map"]
per_class_df = per_class_dfs["best_map"]

print(f"\nValidated on {len(test_dataset)} / {len(test_dataset)} test images.")


# Part 16: Inference speed benchmark for both checkpoints


In [ ]:
@torch.no_grad()
def benchmark_model(model, sample, device, runs=100, warmup=20):
    model.eval()
    sample = sample.to(device)

    for _ in range(warmup):
        _ = model(sample)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.time()
    for _ in range(runs):
        _ = model(sample)
    if device.type == "cuda":
        torch.cuda.synchronize()

    total = time.time() - start
    ms_per_image = (total / runs) * 1000
    fps = runs / total
    return ms_per_image, fps

sample = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
benchmark_rows = []

for tag, cfg in CHECKPOINT_MODELS.items():
    ms, fps = benchmark_model(cfg["model"], sample, device)
    benchmark_rows.append({
        "tag": tag,
        "label": cfg["label"],
        "ms_per_image": ms,
        "fps": fps,
    })
    print(f"{cfg['label']}: {ms:.2f} ms/image | {fps:.2f} FPS")

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_csv = os.path.join(LOG_DIR, f"inference_benchmark_{RUN_TIMESTAMP}.csv")
benchmark_df.to_csv(benchmark_csv, index=False)
print("Saved benchmark CSV to:", benchmark_csv)
benchmark_df


# Part 17: Visualise test predictions for both checkpoints


In [ ]:
def denormalize_img(img_tensor):
    mean = torch.tensor(IMAGENET_MEAN, device=img_tensor.device).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=img_tensor.device).view(3, 1, 1)
    img = img_tensor * std + mean
    return img.clamp(0, 1)

def draw_gt_and_pred_boxes(img_tensor, target, pred, class_names):
    img_tensor = denormalize_img(img_tensor)
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = (img * 255).astype(np.uint8).copy()
    h, w = img.shape[:2]

    # Ground truth in green
    gt_boxes = target["boxes"]
    gt_labels = target["labels"]
    if torch.is_tensor(gt_boxes):
        gt_boxes = gt_boxes.cpu().numpy()
    if torch.is_tensor(gt_labels):
        gt_labels = gt_labels.cpu().numpy()

    for box, cls_id in zip(gt_boxes, gt_labels):
        cx, cy, bw, bh = box
        x1 = int((cx - bw / 2) * w)
        y1 = int((cy - bh / 2) * h)
        x2 = int((cx + bw / 2) * w)
        y2 = int((cy + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        txt = f"GT:{class_names[int(cls_id)]}"
        cv2.putText(img, txt, (x1, max(0, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 2)

    # Predictions in red
    boxes = pred["boxes"]
    scores = pred["scores"]
    labels = pred["labels"]

    for box, score, cls_id in zip(boxes, scores, labels):
        x1 = int(box[0] * w)
        y1 = int(box[1] * h)
        x2 = int(box[2] * w)
        y2 = int(box[3] * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        txt = f"PR:{class_names[int(cls_id)]} {score:.2f}"
        cv2.putText(img, txt, (x1, min(h - 5, y2 + 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 2)
    return img


In [ ]:
def render_sample_predictions_for_model(model, label, images, targets, names, save_path=None):
    model.eval()
    images_dev = images.to(device)

    with torch.no_grad():
        outputs = model(images_dev)

    preds = decode_predictions(
        outputs,
        conf_thresh=EVAL_CONF_THRESH,
        nms_iou=EVAL_NMS_IOU,
        num_classes=NUM_CLASSES
    )

    fig = plt.figure(figsize=(15, 10))
    for i in range(min(4, len(images))):
        plt.subplot(2, 2, i + 1)
        vis = draw_gt_and_pred_boxes(images[i], targets[i], preds[i], CLASS_NAMES)
        plt.imshow(vis)
        plt.title(names[i])
        plt.axis("off")
    plt.suptitle(label)
    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved sample overlay figure to:", save_path)

    plt.show()
    plt.close(fig)

images, targets, names = next(iter(test_loader))

for tag, cfg in CHECKPOINT_MODELS.items():
    sample_path = os.path.join(
        ARTIFACT_DIRS[tag]["blind_dir"],
        f"sample_test_overlays_{tag}_{RUN_TIMESTAMP}.png"
    )
    render_sample_predictions_for_model(
        cfg["model"],
        cfg["label"],
        images,
        targets,
        names,
        save_path=sample_path,
    )


# Part 18: Collect full-test predictions for both checkpoints


In [ ]:
@torch.no_grad()
def run_full_inference(model, loader, device, num_classes):
    model.eval()
    all_preds = []
    all_targets = []
    all_names = []

    for images, targets, names in loader:
        images = images.to(device)
        outputs = model(images)
        preds = decode_predictions(
            outputs,
            conf_thresh=EVAL_CONF_THRESH,
            nms_iou=EVAL_NMS_IOU,
            num_classes=num_classes
        )
        all_preds.extend(preds)
        all_targets.extend(targets)
        all_names.extend(names)

    return all_preds, all_targets, all_names

inference_results = {}

for tag, cfg in CHECKPOINT_MODELS.items():
    preds, targets_full, names_full = run_full_inference(cfg["model"], test_loader, device, NUM_CLASSES)
    assert len(preds) == len(test_dataset) == len(targets_full) == len(names_full), (
        f"Full test inference for {tag} did not cover the full test set."
    )
    inference_results[tag] = {
        "preds": preds,
        "targets": targets_full,
        "names": names_full,
    }
    print(f"{tag}: collected full-test predictions for {len(preds)} images")

# canonical aliases for downstream compatibility
all_preds = inference_results["best_map"]["preds"]
all_targets = inference_results["best_map"]["targets"]
all_names = inference_results["best_map"]["names"]

len(all_preds), len(all_targets), len(all_names)


# Part 19: Failure-case mining for both checkpoints


In [ ]:
failure_dfs = {}

for tag, result in inference_results.items():
    failure_rows = []
    for name, pred, tgt in zip(result["names"], result["preds"], result["targets"]):
        num_gt = len(tgt["boxes"])
        num_pred = len(pred["boxes"])
        failure_rows.append({
            "image": name,
            "gt_boxes": num_gt,
            "pred_boxes": num_pred,
            "difference": num_pred - num_gt,   # negative = missed objects, positive = FPs
        })

    failure_df = pd.DataFrame(failure_rows).sort_values("difference")
    failure_dfs[tag] = failure_df

    failure_csv = os.path.join(
        ARTIFACT_DIRS[tag]["blind_dir"],
        f"failure_cases_{tag}_{RUN_TIMESTAMP}.csv"
    )
    failure_df.to_csv(failure_csv, index=False)

    print(f"\n[{tag}] Most under-predicted (missed objects, highest FN):")
    print(failure_df.head(5)[["image", "gt_boxes", "pred_boxes", "difference"]].to_string(index=False))
    print()
    print(f"[{tag}] Most over-predicted (false positives, highest FP):")
    print(
        failure_df.tail(5).sort_values("difference", ascending=False)[
            ["image", "gt_boxes", "pred_boxes", "difference"]
        ].to_string(index=False)
    )
    print("Saved failure-case CSV to:", failure_csv)

failure_df = failure_dfs["best_map"]
failure_df


# Part 20: Display worst failure cases for both checkpoints


In [ ]:
def get_image_by_name(dataset, target_name):
    idx = dataset.images.index(target_name)
    img, tgt, name = dataset[idx]
    return img, tgt, name

for tag, cfg in CHECKPOINT_MODELS.items():
    failure_df = failure_dfs[tag]
    result = inference_results[tag]

    under_cases = failure_df.head(3)["image"].tolist()
    over_cases  = failure_df.tail(3).sort_values("difference", ascending=False)["image"].tolist()
    show_cases  = under_cases + over_cases
    titles      = [f"UNDER {n}" for n in under_cases] + [f"OVER {n}" for n in over_cases]

    fig = plt.figure(figsize=(16, 12))
    for i, (name, title) in enumerate(zip(show_cases, titles)):
        img, tgt, _ = get_image_by_name(test_dataset, name)
        pred = result["preds"][result["names"].index(name)]
        vis = draw_gt_and_pred_boxes(img, tgt, pred, CLASS_NAMES)
        plt.subplot(3, 2, i + 1)
        plt.imshow(vis)
        plt.title(title, fontsize=8)
        plt.axis("off")

    plt.suptitle(
        f"{cfg['label']} — top failure cases: UNDER=missed objects  OVER=false positives",
        fontsize=10
    )
    plt.tight_layout()

    failure_fig = os.path.join(
        ARTIFACT_DIRS[tag]["blind_dir"],
        f"worst_failures_{tag}_{RUN_TIMESTAMP}.png"
    )
    fig.savefig(failure_fig, dpi=150, bbox_inches="tight")
    print("Saved failure-case figure to:", failure_fig)

    plt.show()
    plt.close(fig)


# Part 21: Blind test analysis summaries for both checkpoints


In [ ]:
blind_summaries = {}

for tag, metrics in test_results.items():
    summary_rows = []
    for cls_id, stats in metrics["per_class"].items():
        summary_rows.append({
            "Class": CLASS_NAMES[cls_id],
            "GT": stats.get("GT", stats.get("GT Count", 0)),
            "TP": stats["TP"],
            "FP": stats["FP"],
            "FN": stats["FN"],
            "AP": stats.get("AP", stats.get("AP@0.5", 0.0)),
            "AP@0.5": stats.get("AP@0.5", stats.get("AP", 0.0)),
            "Precision": stats["Precision"],
            "Recall": stats["Recall"],
            "F1": stats.get("F1", 0.0),
        })

    blind_df_variant = pd.DataFrame(summary_rows)
    blind_summaries[tag] = blind_df_variant

    blind_csv = os.path.join(
        ARTIFACT_DIRS[tag]["blind_dir"],
        f"blind_test_summary_{tag}_{RUN_TIMESTAMP}.csv"
    )
    blind_df_variant.to_csv(blind_csv, index=False)
    print(f"Saved blind test summary for {tag} to:", blind_csv)

blind_df = blind_summaries["best_map"]
blind_df


In [ ]:
for tag, cfg in CHECKPOINT_MODELS.items():
    blind_df_variant = blind_summaries[tag]

    fig = plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.bar(blind_df_variant["Class"], blind_df_variant["AP"])
    plt.title(f"AP per Class — {tag}")
    plt.ylabel("AP")

    plt.subplot(1, 2, 2)
    x = np.arange(len(blind_df_variant))
    width = 0.25
    plt.bar(x - width, blind_df_variant["TP"], width, label="TP")
    plt.bar(x, blind_df_variant["FP"], width, label="FP")
    plt.bar(x + width, blind_df_variant["FN"], width, label="FN")
    plt.xticks(x, blind_df_variant["Class"])
    plt.title(f"Blind Test Error Breakdown — {tag}")
    plt.legend()

    plt.tight_layout()

    plot_path = os.path.join(
        ARTIFACT_DIRS[tag]["blind_dir"],
        f"blind_test_plots_{tag}_{RUN_TIMESTAMP}.png"
    )
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    print("Saved blind-test plots to:", plot_path)

    plt.show()
    plt.close(fig)


# Part 22: Save prediction images for both checkpoints


In [ ]:
saved_manifests = {}

for tag, cfg in CHECKPOINT_MODELS.items():
    save_dir = ARTIFACT_DIRS[tag]["pred_dir"]
    os.makedirs(save_dir, exist_ok=True)

    saved_rows = []
    saved_count = 0

    cfg["model"].eval()
    for images, targets, names in test_loader:
        images_dev = images.to(device)
        with torch.no_grad():
            outputs = cfg["model"](images_dev)
        preds = decode_predictions(
            outputs,
            conf_thresh=EVAL_CONF_THRESH,
            nms_iou=EVAL_NMS_IOU,
            num_classes=NUM_CLASSES
        )

        for img_tensor, target, pred, name in zip(images, targets, preds, names):
            vis = draw_gt_and_pred_boxes(img_tensor, target, pred, CLASS_NAMES)
            out_path = os.path.join(save_dir, name)
            Image.fromarray(vis).save(out_path)
            saved_rows.append({
                "model_tag": tag,
                "image_name": name,
                "saved_path": out_path,
                "num_preds": len(pred["boxes"]),
                "num_gt": len(target["boxes"])
            })
            saved_count += 1

    save_manifest = os.path.join(LOG_DIR, f"prediction_manifest_{tag}_{RUN_TIMESTAMP}.csv")
    pd.DataFrame(saved_rows).to_csv(save_manifest, index=False)
    saved_manifests[tag] = save_manifest

    assert saved_count == len(test_dataset), (
        f"Saved {saved_count} prediction images for {tag}, expected {len(test_dataset)}"
    )
    print(f"[{tag}] Saved prediction images: {saved_count} / {len(test_dataset)}")
    print(f"[{tag}] Saved GT+prediction overlay examples to: {save_dir}")
    print(f"[{tag}] Saved prediction manifest to: {save_manifest}")


# Part 23: Display first 60 best mAP prediction images in the notebook

In [ ]:
import math

display_count = min(60, len(test_dataset))
best_map_pred_dir = ARTIFACT_DIRS["best_map"]["pred_dir"]
gallery_dir = os.path.join(ARTIFACT_DIRS["best_map"]["blind_dir"], "best_map_gallery_sheets")
os.makedirs(gallery_dir, exist_ok=True)

gallery_names = sorted([
    f for f in os.listdir(best_map_pred_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])[:display_count]

images_per_page = 10
num_pages = math.ceil(len(gallery_names) / images_per_page)

print(f"Displaying {len(gallery_names)} best-mAP prediction images in the notebook.")
print("Gallery sheets saved to:", gallery_dir)

for page_idx in range(num_pages):
    batch_names = gallery_names[page_idx * images_per_page:(page_idx + 1) * images_per_page]
    rows = math.ceil(len(batch_names) / 2)

    fig = plt.figure(figsize=(16, 3.5 * rows))
    for plot_idx, name in enumerate(batch_names, start=1):
        img_path = os.path.join(best_map_pred_dir, name)
        vis = np.array(Image.open(img_path).convert("RGB"))
        plt.subplot(rows, 2, plot_idx)
        plt.imshow(vis)
        plt.title(name, fontsize=8)
        plt.axis("off")

    start_idx = page_idx * images_per_page + 1
    end_idx = page_idx * images_per_page + len(batch_names)
    plt.suptitle(f"Best mAP@0.5 predictions — images {start_idx}-{end_idx}", fontsize=12)
    plt.tight_layout()

    gallery_sheet_path = os.path.join(
        gallery_dir,
        f"best_map_gallery_page_{page_idx + 1:02d}_{RUN_TIMESTAMP}.png"
    )
    fig.savefig(gallery_sheet_path, dpi=150, bbox_inches="tight")
    print("Saved gallery sheet to:", gallery_sheet_path)

    plt.show()
    plt.close(fig)


In [ ]:
# Final export cell: package current run and create downloadable ZIPs in /kaggle/working

import os
import json
import glob
import shutil
import torch

assert 'RUN_DIR' in globals(), "RUN_DIR is not defined. Run the training cell first."
assert os.path.isdir(RUN_DIR), f"RUN_DIR not found: {RUN_DIR}"

CKPT_DIR = os.path.join(RUN_DIR, "checkpoints")
LOG_DIR = os.path.join(RUN_DIR, "logs")
PRED_DIR = os.path.join(RUN_DIR, "predictions")
BLIND_DIR = os.path.join(RUN_DIR, "blind_test_analysis")

assert os.path.isdir(CKPT_DIR), f"Missing checkpoints folder: {CKPT_DIR}"
assert os.path.isdir(LOG_DIR), f"Missing logs folder: {LOG_DIR}"
assert os.path.isdir(PRED_DIR), f"Missing predictions folder: {PRED_DIR}"

map_candidates = sorted(glob.glob(os.path.join(CKPT_DIR, "best_map_*.pth")))
map_pfloor_candidates = sorted(glob.glob(os.path.join(CKPT_DIR, "best_map_pfloor_*.pth")))
all_candidates = sorted(glob.glob(os.path.join(CKPT_DIR, "*.pth")))
best_candidates = map_candidates or map_pfloor_candidates or all_candidates
assert best_candidates, f"No checkpoint files found in {CKPT_DIR}"
best_path = best_candidates[0]
print(f"Export using primary checkpoint: {os.path.basename(best_path)}")

EXPORT_DIR = os.path.join(RUN_DIR, "export_bundle")
os.makedirs(EXPORT_DIR, exist_ok=True)

# Copy primary model and keep all checkpoints in the copied checkpoint folder.
shutil.copy2(best_path, os.path.join(EXPORT_DIR, os.path.basename(best_path)))

for sub_name, src_dir in [
    ("logs", LOG_DIR),
    ("predictions", PRED_DIR),
    ("checkpoints", CKPT_DIR),
]:
    dst_dir = os.path.join(EXPORT_DIR, sub_name)
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)

if os.path.isdir(BLIND_DIR):
    dst_dir = os.path.join(EXPORT_DIR, "blind_test_analysis")
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(BLIND_DIR, dst_dir)

summary = {
    "run_timestamp": RUN_TIMESTAMP,
    "run_dir": RUN_DIR,
    "primary_best_model_path": best_path,
    "best_map_path": best_map_path if 'best_map_path' in globals() else None,
    "best_map_pfloor_path": best_map_pfloor_path if 'best_map_pfloor_path' in globals() else None,
    "min_precision_for_map_ckpt": MIN_PRECISION_FOR_MAP_CKPT if 'MIN_PRECISION_FOR_MAP_CKPT' in globals() else None,
    "train_size": len(train_dataset) if 'train_dataset' in globals() else None,
    "val_size": len(val_dataset) if 'val_dataset' in globals() else None,
    "test_size": len(test_dataset) if 'test_dataset' in globals() else None,
    "eval_conf_thresh": EVAL_CONF_THRESH if 'EVAL_CONF_THRESH' in globals() else None,
    "eval_nms_iou": EVAL_NMS_IOU if 'EVAL_NMS_IOU' in globals() else None,
}

checkpoint_summaries = {}
for label, ckpt_path in {
    "best_map": summary["best_map_path"],
    "best_map_pfloor": summary["best_map_pfloor_path"],
}.items():
    if ckpt_path and os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, map_location="cpu")
            checkpoint_summaries[label] = {
                "path": ckpt_path,
                "epoch": ckpt.get("epoch", None),
                "metrics": ckpt.get("metrics", {}),
            }
        except Exception as e:
            checkpoint_summaries[label] = {
                "path": ckpt_path,
                "checkpoint_read_error": str(e),
            }

summary["checkpoint_summaries"] = checkpoint_summaries

summary_json = os.path.join(EXPORT_DIR, f"run_summary_{RUN_TIMESTAMP}.json")
with open(summary_json, "w") as f:
    json.dump(summary, f, indent=2)

summary_txt = os.path.join(EXPORT_DIR, f"run_summary_{RUN_TIMESTAMP}.txt")
with open(summary_txt, "w") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

ZIP_BASE = f"/kaggle/working/run_{RUN_TIMESTAMP}"
ZIP_PATH = shutil.make_archive(ZIP_BASE, "zip", RUN_DIR)

FIXED_ZIP_BASE = "/kaggle/working/final_export"
FIXED_ZIP_PATH = shutil.make_archive(FIXED_ZIP_BASE, "zip", RUN_DIR)

print("RUN_DIR =", RUN_DIR)
print("EXPORT_DIR =", EXPORT_DIR)
print("ZIP_PATH =", ZIP_PATH)
print("FIXED_ZIP_PATH =", FIXED_ZIP_PATH)
print("ZIP EXISTS =", os.path.exists(ZIP_PATH), os.path.exists(FIXED_ZIP_PATH))
print("ZIP SIZE (bytes) =", os.path.getsize(ZIP_PATH), os.path.getsize(FIXED_ZIP_PATH))
print("Save notebook version after this cell finishes. The file /kaggle/working/final_export.zip is the easiest one to download from Kaggle Output.")
